In [0]:
fineguard.gold.high_value_transaction_alert# bootstrap_servers = 'pkc-619z3.us-east1.gcp.confluent.cloud:9092'
# api_key='2GFYDPHLYM7H4GEW'
# api_secret='cflt2Mls9d8EAnp/PnNiYWNefT5G/OCTaogxGAqywMr9bl2+ll6zS9pKCkSOnRkA'
# topic = 'credit_card_transactions'

In [0]:
import json
kafka_connection_json = dbutils.secrets.get(scope = 'fineguard-scope', key = 'kafka_connection_details')
kafka_config = json.loads(kafka_connection_json)
bootstrap_servers = kafka_config['bootstrap_servers']
api_key = kafka_config['api_key']
api_secret = kafka_config['api_secret']
topic = kafka_config['topic']

In [0]:
# Define the shaded login string right above spark.read
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

# Print it to verify it starts with "kafkashaded."
# print(jaas_config)

In [0]:


sample_batch = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)




In [0]:
sample_batch.count()

In [0]:
# display(sample_batch)

In [0]:
from pyspark.sql.functions import col

parse_batch = sample_batch.select(
    col('key').cast('string'),
    col("value").cast("string"),
    col('partition'),
    col('topic'),
    col('timestamp').cast('timestamp'),
    col('offset')
    
)

In [0]:
# display(parse_batch)

In [0]:
parse_batch.write.saveAsTable('fineguard.bronze.transaction_batch_test')

In [0]:


sample_streaming = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)




In [0]:
from pyspark.sql.functions import col

parse_streaming = sample_streaming.select(
    col('key').cast('string'),
    col("value").cast("string"),
    col('partition'),
    col('topic'),
    col('timestamp').cast('timestamp'),
    col('offset')
    
)

In [0]:
streaming_data = parse_streaming.writeStream.format("delta").outputMode("Append")\
    .option('checkpointLocation', '/Volumes/fineguard/source/transactions/checkpoint/')\
    .trigger(availableNow=True)\
    .toTable('fineguard.bronze.transaction_streaming_test')

print("Query id is :",streaming_data.id)

In [0]:
%sql
select count(*) from fineguard.bronze.transaction_streaming_test